In [4]:
import numpy as np
from numpy import random
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
def propensity(k,*args):
    '''Calculate the propersity of a reaction 
    
    parameters
    ----------
    k:int
      reaction constant
    *args:int(s)
      molecular number of reactant(s)
    
    Returns
    -------
    out: int
       propensity of a reaction
    '''
    
    for i in args:
        k=k*i
    return k


def reaction1(a,b,c):
    '''the binding of ribosome into RNA
       Calculate the molecular number change of this reaction
       
    parameters
    ----------
    a:int
      molecular number of first reactant (free ribosome)
    b:int
      molecular number of second reactant (RBS on mRNA)
    c:int
      molecular number of product (Ribosome-RNA_RBS complex)
      
    Returns
    -------
    out: int
       molecular numbers of reactants and product after the reaction has occurred once
    '''
    
    if a>0 and b>0:
        a-=1
        b-=1
        c+=1
    return a,b,c


def reaction2(a,b,c):
    '''Ribosome read through the second codon, release the RBS of mRNA
    Calculate the molecular number change of this reaction type
    
    parameters
    ----------
    a:int
      molecular number of first reactant (Ribosome-RNA_RBS complex)
    b:int
      molecular number of second reactant (the second codon on mRNA)
    c:int
      molecular number of product (RBS on mRNA)
      
    Returns
    -------
    out: int
       molecular numbers of reactants and product after the reaction has occurred once
    
    '''
    
    if a>0:
        a-=1
        b+=1
        c+=1
    return a,b,c


def reaction3(a,b):
    '''ribosome read through mRNA, actually the transition of complex
    or encounter the stop codon, release the free ribsome
    Calculate the molecular number change of this reaction type
    
    parameters
    ----------
    a:int
      molecular number of reactant (Ribosome-RNA complex[i])
    b:int
      molecular number of product (Ribosome-RNA complex[i+1] or free ribosome)
    
    Returns
    -------
    out: int
       molecular numbers of reactant and product after the reaction has occurred once
    
    '''
    
    if a>0:
        a-=1
        b+=1
    return a,b


def translation(sequence):  
    '''Given a mRNA sequence, translate the sequence into amino acid sequence
    
    parameters
    ----------
    sequence:str
      the given mRNA sequence
    
    Returns
    -------
    out: tuple with 2 elements
        first element: int
            how many amino acids would be translated from the given mRNA sequence
        second element: list
            store the amino acid sequence translated from the given mRNA sequence
    '''
    
    codon_dict={'UUU':'F', 'UUC':'F', 'UUA':'L', 'UUG':'L',           # the codon dictionary
                 'UCU':'S', 'UCC':'S', 'UCA':'S', 'UCG':'S',     
                 'UAU':'Y', 'UAC':'Y', 'UAA':'STOP', 'UAG':'STOP',     
                 'UGU':'C', 'UGC':'C', 'UGA':'STOP', 'UGG':'W',     
                 'CUU':'L', 'CUC':'L', 'CUA':'L', 'CUG':'L',     
                 'CCU':'P', 'CCC':'P', 'CCA':'P', 'CCG':'P',     
                 'CAU':'H', 'CAC':'H', 'CAA':'Q', 'CAG':'Q',     
                 'CGU':'R', 'CGC':'R', 'CGA':'R', 'CGG':'R',     
                 'AUU':'I', 'AUC':'I', 'AUA':'I', 'AUG':'M',     
                 'ACU':'T', 'ACC':'T', 'ACA':'T', 'ACG':'T',     
                 'AAU':'N', 'AAC':'N', 'AAA':'K', 'AAG':'K',     
                 'AGU':'S', 'AGC':'S', 'AGA':'R', 'AGG':'R',     
                 'GUU':'V', 'GUC':'V', 'GUA':'V', 'GUG':'V',     
                 'GCU':'A', 'GCC':'A', 'GCA':'A', 'GCG':'A',     
                 'GAU':'D', 'GAC':'D', 'GAA':'E', 'GAG':'E',    
                 'GGU':'G', 'GGC':'G', 'GGA':'G', 'GGG':'G'}
    
    aa_list=[]    # will be used to store the amino acid sequence translated from the mRNA sequence
    
    start_point=sequence.find('AUG')  # search the start codon in the mRNA sequence
    trimmed_sequence=sequence[start_point:]  # trim the mRNA sequence from the start codon
    
    codons_list = [trimmed_sequence[i:i+3] for i in range(0, len(trimmed_sequence), 3)] # trim the sequence into codon list
    
    stop_list=['UAA', 'UAG','UGA']   # stop codon list
    
    n=0
    for i in codons_list:
        if len(i)==1 or len(i)==2:  # means the length of last element is less than 3, not a codon, just pass
            pass
        elif i not in stop_list:  # the codon is not stop codon, translate it into amino acid, and store into aa_list
            n+=1
            aa_list.append(codon_dict[i])
        else:   # the codon is stop codon, break
            break
            
    return n,aa_list
    

def directmethod(sequence, initialcomp_list, reconstant_list, total_rib, rna, t1, steps): 
    '''Use the Direct Method to simulate one reaction step of the translation reaction in Arkin et al 
    
    parameters
    ----------
    sequence: str
        the given mRNA sequence
    initialcomp_list: list
        list that stores the initial molecular number of the transitional Ribosome-RNA[i] complex
    reconstant_list: list
        list that stores reaction constants of all the reactions (values will not change as reaction occur)
    total_rib: int
        initial value of the free ribosome
    rna: int
        initial molecular number of mRNA 
    t1: int
        simulation initial time
    steps: list
        list that store the indexes of codons that have been occupied by Ribosome-RNA[i] complex
        
    Returns
    -------
    out: tuple with 6 elements
       first element: list
            update the No. of Ribosome-RNA[i] complex after one step
       second element: int
            update the No. of free ribosome after one step
       third element: int
            update the No. of free mRNA-RBS after one step
       forth element: float
            update the time after one step
       fifth element: str
            the translated amino acid by this reaction step
       sixth element: int
            the selected codon position which the reaction will occur
    '''        
    
    
    complex_list=initialcomp_list  # get the copy of initialcomp_list,total_rib,rna as the initial value
    rib=total_rib
    rna_rbs=rna
    
    s=translation(sequence) # translate the given mRNA sequence
    n=s[0]  # the number of amino acids that can be translated from the mRNA sequence
    aa_list=s[1] # the amino acids that can be translated
    
    prop_list=[]  
    prop1=propensity(reconstant_list[0],rib,rna_rbs) # calculate the propensity of ribosome binding into mRNA-RBS
    
    prop_list.append(prop1)
         
    for i in range(1,n): # calculate the propensity of ribosome reading through the mRNA
        prop_list.append(propensity(reconstant_list[i],complex_list[i-1]))  
    
    if steps!=[]:  # steps stores the occupied codon positions, exclude them by resetting their propensities to 0
        new_prop = []   # will be used to store the propensities of reactions occurred in available codon positions
        for i, p in enumerate(prop_list):  # get the indexes and values of elements in prop_list
            if i not in steps: # if the index of element is not in steps, it means that the position is available
                new_prop.append(p)  # just store the value into new_prop list
            else:             # else means the index of element is in steps, it means that the position has been occupied
                new_prop.append(0) # just append 0 into the new_prop (means the reaction cannnot occur)
    else:
        new_prop = prop_list # else means steps is [](empty), no position has been occupied, new_prop is equal to prop_list
    
    
    sumprop=sum(new_prop)
    prob_list=[]

    if sumprop > 0:
        for i in new_prop: 
            prob_list.append(i/sumprop)   # calculate the probablity of each reaction
    
    if sum(prob_list)!=0:
        u=random.choice(np.arange(0,n),p=prob_list)  # choose u 
        tau=random.exponential(1/sumprop)     # choose time tau
        
        if u==0:   # u==0, means ribosome binding into mRNA, update the number of ribosome, RNA_RBS and Ribosome-RNA_RBS complex
            rib,rna_rbs,complex_list[0]=reaction1(rib,rna_rbs,complex_list[0])  
            
        elif u==1: # u==1, means ribosome read through the second codon and release the RBS
            complex_list[0],complex_list[1],rna_rbs=reaction2(complex_list[0],complex_list[1],rna_rbs) # release the RBS
            
        elif u==n-1: # u==n-1, means the ribosome encounters the STOP codon
            complex_list[u-1],rib=reaction3(complex_list[u-1],rib) # encouter the STOP codon
            
        else: # else means the ribosome reads through other codon positions, regarded as the transition 
            # of Ribosome-RNA complex[i]-->Ribosome-RNA complex[i+1]
            complex_list[u-1],complex_list[u]=reaction3(complex_list[u-1],complex_list[u]) 
        
    newcomplex_list=[]  
    for i in range(len(complex_list)):  # get and store the updated number of Ribosome-RNA complex[i] into newcomplex_list
        newcomplex_list.append(complex_list[i]) 
        
    rib1=rib  # get the updated number of free ribosome
    rna_rbs1=rna_rbs  # get the updated number of RNA_RBS
    
    aa=aa_list[u]  # get the amino acid translated from this step of reaction
                
    return newcomplex_list, rib1, rna_rbs1, tau, aa, u
        
        

def simulation(sequence,initialcomp_list,reconstant_list,total_rib,rna,t1,t2):
    '''Stimulate the translation reactions during a given time period from t1 to t2
    
    parameters
    ----------
    sequence: str
        the given mRNA sequence
    initialcomp_list: list
        list that stores the initial molecular number of the transitional Ribosome-RNA[i] complex
    reconstant_list: list
        list that stores reaction constants of all the reactions (values will not change as reaction occur)
    total_rib: int
        initial value of the free ribosome
    rna: int
        initial molecular number of mRNA 
    t1: int
        simulation initial time
    t2: int
        simulation ending time
    
    Returns
    -------
    out: tuple with 2 elements
       first element:the dictionary that store the time and positions of each individual reaction
       second element:the dictionary that store the translated amino acid sequence of each individual reaction   
    '''
    
    t0=t1   # use t0 to store the initial value of t1 (stimulation starting time)
    t1_list=[]  # will be used to store the updating reaction time t1
    
    complex_list=initialcomp_list # get the copy of initialcomp_list, free ribosome and mRNA (mRNA-RBS)
    rib=total_rib   
    rna_rbs=rna  
    
    result={}   # will be used to store the time and positions of each individual reaction
    aa_result={} # will be used to store the translated amino acid sequence of each individual reaction
    
    ids=0    # will be used as the indicator of new reaction (also as the name of reaction)
    
    u_list=[]  # will be used to store the generated u at each step
    
    steps=[]  # will be used to store the codon postions that has been occupied (reaction cannot occur)

    while t1<t2:
        y=directmethod(sequence,complex_list,reconstant_list, rib,rna_rbs, t1,steps) # run direct method
        u=y[-1]  # get u
        aa=y[-2]  # get the translated amino acid from this step of reaction
        
        
        if u_list==[]:         # at first u_list is empty, 
            complex_list=y[0]  # update the molecular number of Rib-RNA complex[i], free ribosome and RNA_RBS
            rib=y[1]
            rna_rbs=y[2]

            t1+=y[3]   # update the reaction time t1
            t1_list.append(t1)  # store the updated time 
            u_list.append(u)  # store the u generated from this step (then u_list is not empty)
            
            ids+=1  # start a new reaction (ids will be used as the name of new reaction)
            result[ids]=[(t1,u)]  # store the tuple of (reaction time, u) into dictionary (the ids will be the key)
            aa_result[ids]=[aa]  # store the generated amino acid under the key of ids (ids is the name of reaction)
        
        
        else:  # after one step has occurred, then u_list is not empty
            steps = [v[-1][-1] for v in result.values()] # first get the occupied codon positions and store them into steps
            steps = [s - 1 for s in steps if s!=0] + steps # v[-1][-1] is the last codon position of each individual reaction,
                                            # then the previous position is also occupied by this peptide translation reaction
            
            complex_list=y[0]  # update number of Ribosome-RNA[i], free ribosome and RNA-RBS
            rib=y[1]  
            rna_rbs=y[2]

            u_list.append(u)
                
            if u==0:    # if u==0, means this step is as follows: a new peptide translation reaction has occurred
                ids+=1  # use the new ids as the name of this new reaction
                t1 += y[3]  # update the reaction time
                result[ids]=[(t1,u)]  # store the tuple of (reaction time, u) into dictionary (the new ids will be the key)
                aa_result[ids]=[aa] # store the generated amino acid under the key of ids (ids is the name of reaction)
            else:    # if u!=0, means this step is as follows: one of the current peptide reaction is continuing
                t1 += y[3]  # updating the reaction time
                for k, v in result.items(): # then check which reaction is continuing,
                    if v[-1][-1]==u-1:  # v[-1][-1] is the last codon position of each individual reaction, if v[-1][-1]==u-1,
                        result[k].append((t1, u)) # it means this peptide translation reaction is occuring, just append the 
                        aa_result[k].append(aa) # tuple of (reaction time, u) into result dictionary under this key, and also
                        break                   # append the translated amino acid into aa_result dictionary under this key

            
    for k,v in aa_result.items():
        print('Reaction {}: {}'.format(k,''.join(v)))        
    
    return result,aa_result
       


In [6]:
# sequence='ACAUGCUAGAACCGCAUGUACUAGUUAA'
sequence='AUGCUAGAACCGCUAGAACCGCAUGUACAAGUUAACAAGAACCGCUAGAACCGCAUGUACUAGUUAA'

s=translation(sequence)
n=s[0]

initialcomp_list=[]
for i in range(0,n-1):
    initialcomp_list.append(0)

# print(initialcomp_list)

reconstant_list=[1]
for i in range(n-1):
    reconstant_list.append(2)
# print(reconstant_list)


total_rib=5
rna=1

t1=1
t2=10

# x=directmethod(sequence,initialcomp_list,reconstant_list,total_rib,rna)
# print(x[0])
# print(x[1])
# print(x[2])
# print(x[3])
# print(x[4])
# print(x[5])

x=simulation(sequence,initialcomp_list,reconstant_list,total_rib,rna,t1,t2)
# for i in zip(x, y):
#     print(i)
print(x)

Reaction 1: MLEPLEPHVQVNKNR
Reaction 2: MLEPLEPHVQVNK
Reaction 3: MLEPLEPHVQV
Reaction 4: MLEPLEP
({1: [(1.1625400334719231, 0), (1.9868994701096236, 1), (2.1186879556300924, 2), (2.2439451126028596, 3), (2.507303928586648, 4), (2.5296120750407907, 5), (2.842598347082186, 6), (3.1654599671479553, 7), (3.8226414364849663, 8), (3.9400291452200076, 9), (4.608175737276458, 10), (5.507047789326288, 11), (6.415435012862415, 12), (6.604300672823626, 13), (6.800374231769582, 14)], 2: [(2.285990066874412, 0), (3.1053504660828053, 1), (3.51803857982079, 2), (5.124791525420305, 3), (5.215764437791868, 4), (5.5598106372179394, 5), (6.006047317785791, 6), (6.142108722586783, 7), (6.2603070956337055, 8), (6.401825821069935, 9), (7.14559762552102, 10), (7.3173545311113966, 11), (7.513716251725522, 12)], 3: [(4.328510193555516, 0), (5.550746489490837, 1), (5.633501546437822, 2), (6.336334658264906, 3), (6.412140447481362, 4), (6.674139506749599, 5), (7.106734297026569, 6), (7.2280143725159345, 7), (8.